# Scikit-Learn

## Установка и импорт

Обычно достаточно установить библиотеку и несколько вспомогательных пакетов (`numpy`, `pandas`, `matplotlib`):

```bash
pip install scikit-learn
```

В этом ноутбуке используются встроенные датасеты Scikit-Learn, поэтому скачивать данные из интернета не нужно.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn

print("scikit-learn version:", sklearn.__version__)

## 2. Данные в Scikit-Learn: `X` и `y`

Почти все классы Scikit-Learn работают с двумя основными объектами данных:

- `X` — матрица признаков размера $n_{samples} \times n_{features}$;
- `y` — целевая переменная: класс, число или другой ответ.

Здесь $n_{samples}$ — число объектов, а $n_{features}$ — число признаков.

В задачах **обучения с учителем** (*supervised learning*) объект класса модели получает и `X`, и `y`. В задачах **обучения без учителя** (*unsupervised learning*) объект класса алгоритма обычно получает только `X`.

## 3. Основные семейства классов Scikit-Learn

Главное семейство объектов в Scikit-Learn — **оцениватели** (*estimators*). **Оцениватель** — это объект, у которого есть метод `fit()`, позволяющий подогнать параметры оценщика под набор данных. Можно выделить несколько подсемейств классов, которые представляют собой оценщики:

* **Трансформер** (*transformer*) — оцениватель, у которого есть метод `transform()`. Метод `fit()` объекта класса-трансформера оценивает параметры преобразования по данным, а метод `transform()` применяет это преобразование. Примеры: `StandardScaler`, `MinMaxScaler`, `OneHotEncoder`, `PCA`.

* **Предиктор** (*predictor*) — оцениватель, у которого есть метод `predict()`. Метод `fit()` объекта класса-предиктора обучает модель, а метод `predict()` возвращает предсказания для новых объектов. Частные случаи предикторов — **классификаторы** (*classifiers*), **регрессоры** (*regressors*) и **кластеризаторы** (*clusterers*). Примеры: `LogisticRegression`, `KNeighborsClassifier`, `Ridge`, `KMeans`.

* **Мета-оцениватель** (*meta-estimator*) — оцениватель, который внутри использует другие оцениватели. Метод `fit()` объекта класса-мета-оценивателя управляет обучением вложенных объектов. Примеры: `Pipeline`, `GridSearchCV`, `RandomizedSearchCV`, `OneVsRestClassifier`.


### Параметры конструктора и обученные атрибуты

У объекта класса Scikit-Learn есть две важные группы настроек – параметры и гиперпараметры.

**Параметры** или **обученные атрибуты** (*parameters* or *learned attributes*) появляются после вызова метода `fit()` объекта класса, т.е. это переменные, которые "обучаются", т.е. подгоняются под набор данных. В Scikit-Learn такие атрибуты обычно заканчиваются подчёркиванием: `classes_`, `coef_`, `mean_`, `best_params_`.

Это соглашение удобно: всё, что заканчивается на `_`, обычно было вычислено из данных во время обучения.

**Гиперпараметры** (*hyperparameters*) задаются в конструкторе класса до обучения для управления процессом подгонки параметров под данные:

```python
model = LogisticRegression(max_iter=1000, random_state=42)
```

Здесь `max_iter` и `random_state` — параметры конструктора класса `LogisticRegression`.



## 4. Первый датасет: Iris

Для первой демонстрации возьмём классический датасет **Iris**. Нужно определить вид ириса по измерениям чашелистиков (`sepal`) и лепестков (`petal`).

Это задача **классификации** (*classification*): целевая переменная `y` принимает одно из нескольких дискретных значений.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

X = iris.data
y = iris.target

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Классы:", iris.target_names)

X.head()

In [ ]:
pd.Series(y).map(dict(enumerate(iris.target_names))).value_counts()

## 5. Разделение на обучающую и тестовую выборки

Модель нельзя честно оценивать на тех же данных, на которых она обучалась. Поэтому данные делят на две части:

- **обучающая выборка** (*training set*) — на ней вызывается метод `fit()` объекта класса модели;
- **тестовая выборка** (*test set*) — на ней проверяется качество после обучения.

Функция `train_test_split()` возвращает четыре объекта: `X_train`, `X_test`, `y_train`, `y_test`.

Параметр `stratify=y` сохраняет примерно одинаковые доли классов в train и test. Это полезно для классификации, особенно если классы распределены неравномерно.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

## 6. Первая модель: класс `LogisticRegression`

Несмотря на название, класс `LogisticRegression` используется для классификации.

Для бинарной классификации логистическая регрессия оценивает вероятность класса через сигмоиду:

$$
P(y=1 \mid x) = \sigma(w^\top x + b) = \frac{1}{1 + e^{-(w^\top x + b)}}
$$

Для нескольких классов Scikit-Learn использует обобщение этой идеи на многоклассовую классификацию.

Последовательность действий:

1. создать объект класса `LogisticRegression`;
2. вызвать метод `fit()` объекта класса `LogisticRegression` на обучающих данных;
3. вызвать метод `predict()` для тестовых признаков;
4. сравнить `y_pred` с `y_test`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000, random_state=42)

# Метод fit() объекта класса LogisticRegression обучает модель.
model.fit(X_train, y_train)

# Метод predict() объекта класса LogisticRegression возвращает предсказанные классы.
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Атрибуты объекта класса `LogisticRegression`

После вызова метода `fit()` у объекта `model` появились обученные атрибуты.

Например:

- атрибут `classes_` объекта класса `LogisticRegression` хранит метки классов;
- атрибут `coef_` хранит обученные коэффициенты модели;
- атрибут `intercept_` хранит свободные члены;
- атрибут `n_features_in_` хранит число признаков, которое объект увидел при обучении.

In [ ]:
print("classes_:", model.classes_)
print("coef_.shape:", model.coef_.shape)
print("intercept_:", model.intercept_)
print("n_features_in_:", model.n_features_in_)

In [ ]:
model.coef_

В многоклассовой логистической регрессии для Iris модель строит три линейные функции — по одной для каждого класса. Каждую такую функцию можно геометрически воспринимать как гиперплоскость в пространстве признаков. Матрица `model.coef_` содержит нормальные векторы этих трёх гиперплоскостей, а `model.intercept_` — их свободные члены.

## 7. Метрики классификации

**Метрика качества** (*metric*) — это функция, которая сравнивает истинные ответы $y$ и предсказания модели $\hat{y}$.

Для бинарной классификации удобно ввести четыре величины:

* **истинно положительные** (*true positives, TP*) — объекты положительного класса, которые модель правильно отнесла к положительному классу;
* **ложно положительные** (*false positives, FP*) — объекты отрицательного класса, которые модель ошибочно отнесла к положительному классу;
* **истинно отрицательные** (*true negatives, TN*) — объекты отрицательного класса, которые модель правильно отнесла к отрицательному классу;
* **ложно отрицательные** (*false negatives, FN*) — объекты положительного класса, которые модель ошибочно отнесла к отрицательному классу.

**Доля правильных ответов** (*accuracy*) показывает, какая часть всех объектов классифицирована верно:

$$
\text{accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

**Точность** (*precision*) показывает, какая часть объектов, отнесённых моделью к положительному классу, действительно является положительной:

$$
\text{precision} = \frac{TP}{TP + FP}
$$

**Полнота** (*recall*) показывает, какую часть объектов положительного класса модель смогла найти:

$$
\text{recall} = \frac{TP}{TP + FN}
$$

**F1-мера** (*F1-score*) — это гармоническое среднее precision и recall:

$$
F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}
$$

Метод `score()` объекта класса `LogisticRegression` для классификации по умолчанию возвращает accuracy. Однако одной accuracy часто недостаточно: она не показывает, какие именно ошибки совершает модель. Поэтому вместе с accuracy обычно смотрят **матрицу ошибок** (*confusion matrix*) и отчёт с precision, recall и F1-score.


In [ ]:
print("accuracy_score(y_test, y_pred):", accuracy_score(y_test, y_pred))
print("model.score(X_test, y_test):", model.score(X_test, y_test))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=iris.target_names,
    cmap="Blues"
)

plt.style.use("ordevoir-dark")
plt.title("Confusion matrix: LogisticRegression")
plt.show()

## 8. Трансформер: класс `StandardScaler`

Многие алгоритмы чувствительны к масштабу признаков. Например, **метод ближайших соседей** (*k-nearest neighbors*) сравнивает объекты через расстояния. Если один признак измеряется в больших числах, он может доминировать над остальными.

Класс `StandardScaler` стандартизирует каждый признак:

$$
z_j = \frac{x_j - \mu_j}{\sigma_j}
$$

Здесь $x_j$ — значение признака, $\mu_j$ — среднее значение признака на обучающей выборке, $\sigma_j$ — стандартное отклонение признака на обучающей выборке.

Алгоритм работы объекта класса `StandardScaler`:

1. метод `fit()` объекта класса `StandardScaler` вычисляет `mean_` и `scale_` по обучающим данным;
2. метод `transform()` применяет найденные `mean_` и `scale_` к новым данным;
3. метод `fit_transform()` сначала вызывает `fit()`, затем `transform()`.

Важно: средние и стандартные отклонения нужно вычислять только на `X_train`, а затем применять к `X_test`. Иначе возникает **утечка данных** (*data leakage*).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Метод fit() объекта класса StandardScaler вычисляет параметры масштабирования.
scaler.fit(X_train)

# Метод transform() объекта класса StandardScaler применяет преобразование.
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("mean_:", scaler.mean_.round(2))
print("scale_:", scaler.scale_.round(2))
print("Среднее после transform() на train:", X_train_scaled.mean(axis=0).round(6))

## 9. Класс `KNeighborsClassifier`

Класс `KNeighborsClassifier` реализует метод ближайших соседей.

Идея алгоритма:

1. запомнить обучающую выборку;
2. для нового объекта найти $k$ ближайших объектов из обучающей выборки;
3. взять наиболее частый класс среди соседей.

При евклидовом расстоянии расстояние между объектами можно записать так:

$$
d(x, x_i) = \sqrt{\sum_{j=1}^{p}(x_j - x_{ij})^2}
$$

Здесь $p$ — число признаков.

Метод `fit()` объекта класса `KNeighborsClassifier` сохраняет обучающие данные внутри объекта. Метод `predict()` ищет соседей для новых объектов и возвращает классы.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn_manual = knn.predict(X_test_scaled)

print("Accuracy KNN после ручного StandardScaler:", accuracy_score(y_test, y_pred_knn_manual))

## 10. Конвейер: класс `Pipeline`

Класс `Pipeline` объединяет несколько шагов в один объект.

В нашем случае объект класса `Pipeline` будет состоять из двух шагов:

1. объект класса `StandardScaler` — трансформирует признаки;
2. объект класса `KNeighborsClassifier` — обучается и предсказывает классы.

Когда вызывается метод `fit()` объекта класса `Pipeline`, происходит следующее:

1. вызывается метод `fit_transform()` объекта класса `StandardScaler` на `X_train`;
2. результат передаётся в метод `fit()` объекта класса `KNeighborsClassifier`.

Когда вызывается метод `predict()` объекта класса `Pipeline`, происходит следующее:

1. вызывается метод `transform()` уже обученного объекта класса `StandardScaler` на `X_test`;
2. результат передаётся в метод `predict()` объекта класса `KNeighborsClassifier`.

`Pipeline` уменьшает риск утечки тестовых данных в тренировочный процесс и делает весь ML-процесс единым объектом.

In [ ]:
from sklearn.pipeline import Pipeline

knn_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5))
    ]
)

# Метод fit() объекта класса Pipeline обучает все шаги в правильном порядке.
knn_pipeline.fit(X_train, y_train)

# Метод predict() объекта класса Pipeline сам применяет transform() перед предсказанием.
y_pred_knn = knn_pipeline.predict(X_test)

print("Accuracy KNN pipeline:", accuracy_score(y_test, y_pred_knn))

### Атрибуты объекта класса `Pipeline`

У объекта класса `Pipeline` есть удобные атрибуты:

- атрибут `steps` хранит список шагов;
- атрибут `named_steps` позволяет обращаться к шагам по именам;
- внутри `named_steps["scaler"]` лежит обученный объект класса `StandardScaler`;
- внутри `named_steps["knn"]` лежит обученный объект класса `KNeighborsClassifier`.

In [ ]:
print("Имена шагов:", list(knn_pipeline.named_steps.keys()))
print("mean_ у scaler внутри pipeline:", knn_pipeline.named_steps["scaler"].mean_.round(2))
print("n_neighbors у knn внутри pipeline:", knn_pipeline.named_steps["knn"].n_neighbors)

## 11. Кросс-валидация

Один train/test split может оказаться случайно удачным или неудачным. **Кросс-валидация** (*cross-validation*) даёт более устойчивую оценку.

При $k$-fold cross-validation данные делятся на $k$ частей. Затем алгоритм повторяется $k$ раз:

1. одна часть используется как валидационная;
2. остальные $k - 1$ частей используются для обучения;
3. качество записывается;
4. в конце считается среднее качество.

Среднее значение метрики:

$$
\bar{s} = \frac{1}{k}\sum_{i=1}^{k}s_i
$$

Функция `cross_val_score()` получает объект класса модели или объект класса `Pipeline`, несколько раз вызывает его метод `fit()` и оценивает качество.

Когда в `cross_val_score()` передаётся объект класса `Pipeline`, масштабирование заново обучается внутри каждого fold. Это правильное поведение: валидационная часть fold не используется при вычислении `mean_` и `scale_`.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    knn_pipeline,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("CV scores:", scores)
print("Mean accuracy:", scores.mean().round(3))
print("Std:", scores.std().round(3))

## 12. Подбор гиперпараметров: класс `GridSearchCV`

**Гиперпараметры** (*hyperparameters*) — это настройки, которые задаются до обучения и не вычисляются напрямую методом `fit()`.

Например, у объекта класса `KNeighborsClassifier` гиперпараметрами являются:

- `n_neighbors` — число соседей;
- `weights` — способ взвешивания голосов соседей.

Класс `GridSearchCV` перебирает заданную сетку параметров. Для каждой комбинации он запускает кросс-валидацию и выбирает лучшую комбинацию.

Если подбираются параметры шага внутри `Pipeline`, имя параметра записывается через двойное подчёркивание:

```python
"knn__n_neighbors"
```

Здесь `knn` — имя шага в `Pipeline`, а `n_neighbors` — параметр конструктора класса `KNeighborsClassifier`.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"]
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

# Метод fit() объекта класса GridSearchCV запускает перебор и кросс-валидацию.
grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print("Лучшая CV accuracy:", round(grid_search.best_score_, 3))
print("Test accuracy:", round(grid_search.score(X_test, y_test), 3))

### Атрибуты объекта класса `GridSearchCV`

После вызова метода `fit()` у объекта `grid_search` появляются важные атрибуты:

- атрибут `best_params_` хранит лучшую комбинацию гиперпараметров;
- атрибут `best_score_` хранит лучшее среднее качество на кросс-валидации;
- атрибут `best_estimator_` хранит лучший обученный объект класса `Pipeline`;
- атрибут `cv_results_` хранит подробную таблицу всех запусков.

In [ ]:
print("Тип best_estimator_:", type(grid_search.best_estimator_))
print("Шаги best_estimator_:", list(grid_search.best_estimator_.named_steps.keys()))

results = pd.DataFrame(grid_search.cv_results_)

cols = [
    "param_knn__n_neighbors",
    "param_knn__weights",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

results[cols].sort_values("rank_test_score").head(10)

## 13. Мини-пример регрессии: классы `Ridge` и `Pipeline`

**Регрессия** (*regression*) предсказывает число, а не класс.

Для примера используем датасет `diabetes`. Цель — предсказать численный показатель прогрессирования заболевания.

Класс `Ridge` реализует линейную регрессию с $L_2$-регуляризацией. В упрощённом виде задача оптимизации выглядит так:

$$
\min_w \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \lVert w \rVert_2^2
$$

Параметр `alpha` объекта класса `Ridge` управляет силой регуляризации: чем больше `alpha`, тем сильнее штраф за большие коэффициенты.

Метод `fit()` объекта класса `Ridge` обучает коэффициенты, а метод `predict()` возвращает численные предсказания.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score


diabetes = load_diabetes(as_frame=True)
X_reg = diabetes.data
y_reg = diabetes.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42
)

reg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ]
)

reg_pipeline.fit(X_train_reg, y_train_reg)
y_pred_reg = reg_pipeline.predict(X_test_reg)

print("MAE:", round(mean_absolute_error(y_test_reg, y_pred_reg), 2))
print("R²:", round(r2_score(y_test_reg, y_pred_reg), 3))

### Метрики регрессии

**Средняя абсолютная ошибка** (*mean absolute error, MAE*) показывает средний модуль ошибки:

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}\lvert y_i - \hat{y}_i \rvert
$$

**Коэффициент детерминации** (*R-squared, $R^2$*) показывает, насколько модель лучше простого предсказания среднего значения. Значение ближе к $1$ обычно означает лучшее качество.

In [ ]:
plt.scatter(y_test_reg, y_pred_reg, alpha=0.7)
plt.xlabel("Истинное значение")
plt.ylabel("Предсказание")
plt.title("Regression: true vs predicted")

min_value = min(y_test_reg.min(), y_pred_reg.min())
max_value = max(y_test_reg.max(), y_pred_reg.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.show()

## 14. Мини-пример кластеризации: класс `KMeans`

**Кластеризация** (*clustering*) — это обучение без учителя: у нас есть `X`, но нет `y`.

Класс `KMeans` ищет $k$ центров кластеров и относит каждый объект к ближайшему центру.

Целевая функция KMeans:

$$
\min_{C_1, \dots, C_k}\sum_{r=1}^{k}\sum_{x_i \in C_r}\lVert x_i - \mu_r \rVert_2^2
$$

Здесь $C_r$ — кластер, $\mu_r$ — центр кластера.

Метод `fit()` объекта класса `KMeans` ищет центры кластеров. Метод `predict()` относит новые объекты к найденным кластерам. Метод `fit_predict()` сначала обучает объект, а затем сразу возвращает номера кластеров.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

cluster_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=3, random_state=42, n_init="auto"))
    ]
)

clusters = cluster_pipeline.fit_predict(X)

print("Первые 10 кластеров:", clusters[:10])
print("Adjusted Rand Index:", round(adjusted_rand_score(y, clusters), 3))

In [ ]:
# Визуализируем данные в 2D через объект класса PCA.
# PCA здесь используется только для отображения, а не для обучения KMeans.
X_scaled = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, alpha=0.8)
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("KMeans clusters on Iris")
plt.show()

## 15. Типовой рабочий процесс в Scikit-Learn

Базовый ML-процесс удобно объяснять как работу с объектами классов:

1. подготовить `X` и `y`;
2. разделить данные через `train_test_split()`;
3. выбрать классы для предобработки и модели;
4. создать объект класса `Pipeline`;
5. вызвать метод `fit()` объекта класса `Pipeline`;
6. вызвать метод `predict()` или `score()`;
7. проверить качество через метрики;
8. оценить устойчивость через `cross_val_score()`;
9. подобрать гиперпараметры через объект класса `GridSearchCV`;
10. использовать `best_estimator_` как финальный обученный объект.

## 16. Короткая шпаргалка по классам, методам и атрибутам

| Класс / объект | Что демонстрировать | Методы | Атрибуты |
|---|---|---|---|
| `LogisticRegression` | классификация | `fit()`, `predict()`, `score()` | `classes_`, `coef_`, `intercept_` |
| `StandardScaler` | стандартизация признаков | `fit()`, `transform()`, `fit_transform()` | `mean_`, `scale_` |
| `KNeighborsClassifier` | метод ближайших соседей | `fit()`, `predict()`, `score()` | `n_features_in_` |
| `Pipeline` | объединение шагов | `fit()`, `predict()`, `transform()` | `steps`, `named_steps` |
| `GridSearchCV` | подбор гиперпараметров | `fit()`, `predict()`, `score()` | `best_params_`, `best_score_`, `best_estimator_`, `cv_results_` |
| `Ridge` | регрессия | `fit()`, `predict()`, `score()` | `coef_`, `intercept_` |
| `KMeans` | кластеризация | `fit()`, `predict()`, `fit_predict()` | `cluster_centers_`, `labels_` |
| `PCA` | понижение размерности | `fit()`, `transform()`, `fit_transform()` | `components_`, `explained_variance_ratio_` |

Главное правило лекции: первый раз называем полностью — например, «метод `fit()` объекта класса `LogisticRegression`». При повторном упоминании можно говорить короче: «метод `fit()`».

## 17. Идеи для коротких упражнений

1. Поменять `n_neighbors` у объекта класса `KNeighborsClassifier` и посмотреть, как меняется качество.
2. Заменить класс `KNeighborsClassifier` на `RandomForestClassifier` внутри `Pipeline`.
3. Изменить метрику в объекте класса `GridSearchCV`, например на `f1_macro`.
4. Для регрессии подобрать `alpha` у объекта класса `Ridge` через `GridSearchCV`.
5. Убрать объект класса `StandardScaler` из `Pipeline` и сравнить результат для KNN.

Главная мысль: Scikit-Learn ценен не только набором алгоритмов, но и единым интерфейсом классов, методов и атрибутов.